# Standalone ATIF Evaluation with NeMo Agent Toolkit

This notebook demonstrates how to use `nvidia-nat-eval` as a **standalone evaluation
component** — without building or running a NeMo Agent Toolkit workflow.

The key idea: take an [ATIF (Agent Trajectory Interchange Format)](https://github.com/harbor-framework/harbor/blob/main/docs/rfcs/0001-trajectory-format.md)
trajectory produced by *any* agent framework, run built-in evaluators from NeMo Agent Toolkit on it,
and get structured scoring output.

**What this proves:**
- NeMo Agent Toolkit eval can consume ATIF trajectories directly (no `IntermediateStep` conversion)
- Evaluators are initialized via the existing builder and registry system using a programmatic config
- `EvaluationHarness` orchestrates ATIF-native evaluation without any YAML config files

**Requirements:**
- `nvidia-nat-eval` and `nvidia-nat-ragas` (evaluator plugin)
- An NVIDIA API key (`NVIDIA_API_KEY`) for NIM-hosted LLM endpoints

## 1. Install Dependencies

In [ ]:
# Install nat-eval and the RAGAS evaluator plugin.
# These are the only packages needed for standalone ATIF evaluation.
# (nat-core is pulled in as a transitive dependency — no direct interaction required.)
#
# For development on this branch, install from local source:
!uv pip install -q -e ../../packages/nvidia_nat_eval -e ../../packages/nvidia_nat_ragas
#
# For released versions, use:
# !uv pip install nvidia-nat-eval nvidia-nat-ragas

## 2. Define an ATIF Trajectory

This trajectory represents a completed agent run — it could come from any agent
framework that outputs ATIF. Here we define one inline that simulates a simple
RAG-style Q&A interaction.

In [ ]:
import json

atif_trajectory_data = {
    "schema_version": "ATIF-v1.6",
    "session_id": "demo-standalone-eval-001",
    "agent": {
        "name": "external-rag-agent",
        "version": "1.0.0",
        "model_name": "meta/llama-3.3-70b-instruct",
    },
    "steps": [
        {
            "step_id": 1,
            "timestamp": "2026-03-05T10:00:00Z",
            "source": "user",
            "message": "What is LangSmith and how does it help with LLM development?",
        },
        {
            "step_id": 2,
            "timestamp": "2026-03-05T10:00:02Z",
            "source": "agent",
            "message": "Let me search for information about LangSmith.",
            "tool_calls": [
                {
                    "tool_call_id": "call_search_001",
                    "function_name": "knowledge_base_search",
                    "arguments": {"query": "LangSmith LLM development"},
                }
            ],
            "observation": {
                "results": [
                    {
                        "source_call_id": "call_search_001",
                        "content": (
                            "LangSmith is a platform for building production-grade LLM applications. "
                            "It provides tools for debugging, testing, evaluating, and monitoring "
                            "LLM applications. Key features include trace logging, dataset management, "
                            "and evaluation frameworks for systematic testing of LLM outputs."
                        ),
                    }
                ]
            },
            "metrics": {"prompt_tokens": 150, "completion_tokens": 30},
        },
        {
            "step_id": 3,
            "timestamp": "2026-03-05T10:00:05Z",
            "source": "agent",
            "message": (
                "LangSmith is a platform for building production-grade LLM applications. "
                "It helps with LLM development by providing tools for debugging, testing, "
                "evaluating, and monitoring your applications. Key capabilities include "
                "trace logging to understand model behavior, dataset management for "
                "organizing test cases, and evaluation frameworks for systematic testing "
                "of LLM outputs."
            ),
            "metrics": {"prompt_tokens": 250, "completion_tokens": 80},
        },
    ],
    "final_metrics": {
        "total_prompt_tokens": 400,
        "total_completion_tokens": 110,
        "total_steps": 3,
    },
}

print(json.dumps(atif_trajectory_data, indent=2))

## 3. Parse into ATIF Pydantic Models

NeMo Agent Toolkit ships ATIF Pydantic models derived from the
[Harbor reference implementation](https://github.com/harbor-framework/harbor).
We parse the raw dict into a validated `ATIFTrajectory` object.

In [ ]:
from nat.data_models.atif import ATIFTrajectory

trajectory = ATIFTrajectory.model_validate(atif_trajectory_data)

print(f"Schema version: {trajectory.schema_version}")
print(f"Session ID:     {trajectory.session_id}")
print(f"Agent:          {trajectory.agent.name} v{trajectory.agent.version}")
print(f"Steps:          {len(trajectory.steps)}")
print(f"Final metrics:  {trajectory.final_metrics}")

## 4. Build `AtifEvalSample` Objects

`AtifEvalSample` wraps an ATIF trajectory with evaluation metadata — the expected
output (ground truth), actual output, and an item ID. This is the input contract
for ATIF-native evaluators.

In [ ]:
from nat.plugins.eval.evaluator.atif_evaluator import AtifEvalSample

# The expected output is what a perfect agent would produce (ground truth)
expected_output = (
    "LangSmith is a platform for building production-grade LLM applications. "
    "It provides debugging, testing, evaluation, and monitoring capabilities."
)

# The actual output is the agent's final response (last agent step message)
actual_output = trajectory.steps[-1].message

sample = AtifEvalSample(
    item_id="langsmith-q1",
    trajectory=trajectory,
    expected_output_obj=expected_output,
    output_obj=actual_output,
)

atif_samples = [sample]

print(f"Created {len(atif_samples)} AtifEvalSample(s)")
print(f"  item_id:  {sample.item_id}")
print(f"  output:   {str(sample.output_obj)[:80]}...")
print(f"  expected: {str(sample.expected_output_obj)[:80]}...")

## 5. Create a Programmatic `Config` for Evaluator Construction

To use built-in evaluators (RAGAS, trajectory, etc.), we need to
initialize them through the `WorkflowEvalBuilder`. The builder requires a
configuration object.

We construct a minimal configuration programmatically — **no YAML file needed**.
It only contains:
- `llms`: LLM definitions for evaluators that use LLM-as-judge
- `eval.general`: shared evaluation settings (e.g. concurrency)
- `eval.evaluators`: which evaluators to build and their settings

Everything else (workflow, functions, embedders, etc.) uses empty defaults.

In [ ]:
from nat.runtime.loader import PluginTypes
from nat.runtime.loader import discover_and_register_plugins

# Plugin discovery is required before constructing Config objects.
# This registers all evaluator types, LLM types, etc. with the type registry
# so that pydantic discriminated unions resolve correctly.
discover_and_register_plugins(PluginTypes.CONFIG_OBJECT)

In [ ]:
from nat.data_models.config import Config
from nat.utils.data_models.schema_validator import validate_schema

# Build the config as a dict — this mirrors YAML structure but lives in Python.
# Using a dict + validate_schema ensures the discriminated unions resolve correctly.
config_dict = {
    "llms": {
        "eval_llm": {
            "_type": "nim",
            "model_name": "nvidia/llama-3.3-nemotron-super-49b-v1",
            "temperature": 0.0,
        },
    },
    "eval": {
        "general": {
            "max_concurrency": 1,
        },
        "evaluators": {
            "accuracy": {
                "_type": "ragas",
                "llm_name": "eval_llm",
                "metric": "AnswerAccuracy",
                "enable_atif_evaluator": True,
            },
        },
    },
}

config = validate_schema(config_dict, Config)

print(f"LLMs configured:      {list(config.llms.keys())}")
print(f"Evaluators configured: {list(config.eval.evaluators.keys())}")
print(f"Workflow type:         {config.workflow.type} (empty — no workflow needed)")

## 6. Build Evaluators via `WorkflowEvalBuilder`

The builder uses the type registry to resolve evaluator configurations to their
implementations. For RAGAS, this means:
1. Resolve `eval_llm` → NIM LLM client
2. Initialize the RAGAS metric (`AnswerAccuracy`)
3. Build an evaluator that implements the `AtifEvaluator` protocol (when
   `enable_atif_evaluator=True`)
4. Return the evaluator via `get_evaluator(name)`, which can be checked with
   `isinstance(evaluator_info, AtifEvaluator)` to confirm ATIF support

In [ ]:
from nat.plugins.eval.evaluator.atif_evaluator import AtifEvaluator
from nat.plugins.eval.runtime.builder import WorkflowEvalBuilder

# WorkflowEvalBuilder is an async context manager that manages evaluator lifecycle
eval_builder = WorkflowEvalBuilder(
    general_config=config.general,
    eval_general_config=config.eval.general,
)

await eval_builder.__aenter__()
await eval_builder.populate_builder(config, skip_workflow=True)

# Collect evaluators that support the ATIF protocol
atif_evaluators = {}
for name in config.eval.evaluators:
    evaluator_info = eval_builder.get_evaluator(name)
    if isinstance(evaluator_info, AtifEvaluator):
        atif_evaluators[name] = evaluator_info
        print(f"  [ATIF] {name}: {evaluator_info.description}")
    else:
        print(f"  [Legacy] {name}: {evaluator_info.description}")

print(f"\nATIF-native evaluators ready: {list(atif_evaluators.keys())}")

## 7. Run Evaluation via `EvaluationHarness`

The `EvaluationHarness` dispatches ATIF-native evaluators against the sample list,
respecting the `max_concurrency` setting from `eval.general`. It returns a
`dict[str, EvalOutput]` — one entry per evaluator.

In [ ]:
from nat.plugins.eval.runtime.eval_harness import EvaluationHarness

harness = EvaluationHarness()
results = await harness.evaluate(
    evaluators=atif_evaluators,
    atif_samples=atif_samples,
)

print("=" * 60)
print("Evaluation Results")
print("=" * 60)
for evaluator_name, eval_output in results.items():
    print(f"\n--- {evaluator_name} ---")
    print(f"  Average Score: {eval_output.average_score}")
    for item in eval_output.eval_output_items:
        print(f"  Item {item.id}: score={item.score}")
        if item.reasoning:
            reasoning_str = str(item.reasoning)
            print(f"    reasoning: {reasoning_str[:200]}{'...' if len(reasoning_str) > 200 else ''}")

## 8. Cleanup

Release builder resources (LLM connections, etc.).

In [ ]:
await eval_builder.__aexit__(None, None, None)
print("Builder resources released.")

## 9. Summary

This notebook demonstrated:

1. **ATIF as the interoperability contract** — We loaded an ATIF trajectory that
   could have been produced by any agent framework, not just NeMo Agent Toolkit.

2. **Programmatic configuration** — No YAML file was needed. We constructed a
   configuration object in Python with only the fields relevant to evaluation
   (LLMs, eval settings, and evaluators).

3. **Builder-based evaluator construction** — `WorkflowEvalBuilder` resolved evaluator
   configurations through the type registry and constructed fully-initialized
   evaluator instances, including LLM-as-judge setup.

4. **`EvaluationHarness` for standalone scoring** — The harness ran ATIF-native
   evaluators directly on `AtifEvalSample` objects, returning structured `EvalOutput`.

This proves that `nvidia-nat-eval` can serve as a drop-in evaluation component
for any system that produces ATIF trajectories — no NeMo Agent Toolkit workflow required.